# 04  -  Phylogenetic Grouping

**Goal:** Assign each of the 878 genomes to a phylogroup using Mash whole-genome
distances, then define the `GroupedStratifiedKFold` grouping variable for all
classifier phases (7 onward).

**Why this must come before any classifier training:**  
Standard stratified CV assumes all observations are independent. Two near-identical
clones (e.g. two IC2 *A. baumannii* isolates from the same hospital) split across
train/test folds allow the model to "recognise" the test clone as a near-duplicate
of a training clone  -  inflating accuracy without testing generalisation. Phylogenetic
grouped CV prevents this by keeping all members of a clone cluster in the same fold.

**What this notebook produces:**
- `data/interim/mash/`  -  Mash sketch files and pairwise distance matrix
- `data/processed/feature_matrix.parquet`  -  updated with `phylogroup` column
- `data/processed/cv_groups.parquet`  -  grouping series for CV

**Mash (MinHash sketching) in one sentence:**  
Each genome is compressed into a sketch of 1000 minimum hash values drawn from its
set of 21-mer k-mers; the fraction of shared minimums between two sketches estimates
their Jaccard k-mer similarity, which Mash converts to a genomic distance.

## Section 1  -  Imports and paths

In [1]:
import subprocess
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings("ignore")

ROOT     = Path("..")
GENOME_DIR = ROOT / "data" / "raw" / "genomes"
MASH_DIR   = ROOT / "data" / "interim" / "mash"
PROC       = ROOT / "data" / "processed"
FIG_DIR    = ROOT / "results" / "figures" / "phylo"
FIG_DIR.mkdir(parents=True, exist_ok=True)
MASH_DIR.mkdir(parents=True, exist_ok=True)

# Load feature matrix  -  its index defines the 878 accessions to process
fm = pd.read_parquet(PROC / "feature_matrix_3460.parquet")
ACCESSIONS = set(fm.index)          # dot notation: GCF_000418345.1
print(f"Genomes in feature matrix: {len(ACCESSIONS)}")
print(f"Species counts:\n{fm['species'].value_counts().to_string()}")

Genomes in feature matrix: 878
Species counts:
species
saureus        150
paeruginosa    150
abaumannii     150
efaecium       150
ecloaceae      146
kpneumoniae    132


## Section 2  -  Locate genome FASTAs

FASTA files use underscore notation (`GCF_000418345_1.fna`).  
The feature matrix index uses dot notation (`GCF_000418345.1`).  
We convert on the fly and restrict to the 878 matrix accessions,
skipping the 22 MLST-excluded genomes that are present on disk but not in the matrix.

In [2]:
def fasta_stem_to_accession(stem: str) -> str:
    # GCF_000418345_1  →  GCF_000418345.1
    # rsplit on '_' from the right, once, then join with '.'
    parts = stem.rsplit("_", 1)
    return ".".join(parts)

fasta_map = {}   # accession (dot) → Path to FASTA
for fna in GENOME_DIR.rglob("*.fna"):
    stem = fna.stem                              # GCF_000418345_1
    acc  = fasta_stem_to_accession(stem)         # GCF_000418345.1
    if acc in ACCESSIONS:
        fasta_map[acc] = fna

missing = ACCESSIONS - set(fasta_map)
print(f"FASTAs found for matrix accessions: {len(fasta_map)}/878")
if missing:
    print(f"WARNING  -  {len(missing)} accessions missing FASTAs: {list(missing)[:5]}")
else:
    print("All 878 accessions have a matching FASTA.")

FASTAs found for matrix accessions: 878/878
All 878 accessions have a matching FASTA.


## Section 3  -  Mash sketching (k=21, s=1000)

**What `mash sketch` does:**  
For each genome FASTA, it:  
1. Slides a window of k=21 bases across the entire sequence.  
2. Hashes each 21-mer to an integer.  
3. Keeps only the s=1000 smallest hash values  -  the "sketch."

The sketch is ~8 KB regardless of genome size (5 Mb or 7 Mb  -  same sketch size).  
All 878 sketches are collected into a single `.msh` archive for fast pairwise comparison.

`-p 4` uses 4 parallel threads. Runtime: ~2 minutes.

In [3]:
SKETCH_FILE = MASH_DIR / "all_genomes.msh"

if SKETCH_FILE.exists():
    print(f"Sketch file already exists: {SKETCH_FILE}   -  skipping sketching step.")
else:
    fasta_paths = [str(fasta_map[acc]) for acc in sorted(fasta_map)]
    sketch_list_file = MASH_DIR / "fasta_list.txt"
    sketch_list_file.write_text("\n".join(fasta_paths))

    cmd = [
        "/opt/homebrew/Caskroom/miniconda/base/envs/eskape-ml/bin/mash", "sketch",
        "-k", "21",       # k-mer size: validated default for whole-genome bacterial comparison
        "-s", "1000",     # sketch size: 1000 minimums per genome
        "-l",             # read FASTA paths from a list file (not command-line args)
        "-o", str(SKETCH_FILE.with_suffix("")),   # mash appends .msh automatically
        "-p", "4",        # parallel threads
        str(sketch_list_file),
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[:500])
        raise RuntimeError("mash sketch failed")
    print("Sketching complete.")
    print(result.stderr.strip()[:200])

print(f"Sketch file: {SKETCH_FILE}")
print(f"Size: {SKETCH_FILE.stat().st_size / 1e6:.1f} MB")

Sketch file already exists: ../data/interim/mash/all_genomes.msh   -  skipping sketching step.
Sketch file: ../data/interim/mash/all_genomes.msh
Size: 7.2 MB


## Section 4  -  Pairwise Mash distances

`mash dist` compares every sketch against every other sketch.  
878 genomes → 878×877/2 = 384,753 pairs.  

Output format: `genome_A  genome_B  mash_dist  p_value  shared_kmers`  
We only need the first three columns.

Runtime: ~3 minutes.

In [4]:
DIST_FILE = MASH_DIR / "pairwise_distances.tsv"

if DIST_FILE.exists():
    print(f"Distance file already exists: {DIST_FILE}   -  skipping dist step.")
else:
    cmd = [
        "/opt/homebrew/Caskroom/miniconda/base/envs/eskape-ml/bin/mash", "dist",
        "-p", "4",
        str(SKETCH_FILE),
        str(SKETCH_FILE),
    ]
    print("Running:", " ".join(cmd))
    print("(This takes ~3 minutes  -  computing 384,753 pairwise distances)")
    with open(DIST_FILE, "w") as fh:
        result = subprocess.run(cmd, stdout=fh, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[:500])
        raise RuntimeError("mash dist failed")
    print("Distance computation complete.")

n_lines = sum(1 for _ in open(DIST_FILE))
print(f"Distance file: {DIST_FILE}")
print(f"Lines (pairs): {n_lines:,}  (expected 878×878 = {878*878:,} including self-distances)")

Distance file already exists: ../data/interim/mash/pairwise_distances.tsv   -  skipping dist step.
Distance file: ../data/interim/mash/pairwise_distances.tsv
Lines (pairs): 770,884  (expected 878×878 = 770,884 including self-distances)


## Section 5  -  Build the symmetric distance matrix

Mash dist outputs all 878×878 pairs (including self-distances = 0 and both
directions A→B and B→A). We reshape this into a 878×878 symmetric matrix with
accession labels.

In [5]:
def fasta_path_to_accession(path_str: str) -> str:
    # /path/to/genomes/saureus/GCF_000418345_1.fna  →  GCF_000418345.1
    stem = Path(path_str).stem          # GCF_000418345_1
    return fasta_stem_to_accession(stem)

print("Loading distance file...")
dist_df = pd.read_csv(
    DIST_FILE, sep="\t", header=None,
    names=["query", "ref", "dist", "pval", "shared"],
    usecols=["query", "ref", "dist"],
)
print(f"  Rows loaded: {len(dist_df):,}")

# Convert file paths to accession IDs
dist_df["query"] = dist_df["query"].apply(fasta_path_to_accession)
dist_df["ref"]   = dist_df["ref"].apply(fasta_path_to_accession)

# Filter to the 878 matrix accessions (drops any stray rows)
mask = dist_df["query"].isin(ACCESSIONS) & dist_df["ref"].isin(ACCESSIONS)
dist_df = dist_df[mask]

# Pivot to square matrix
acc_list = sorted(ACCESSIONS)
dist_matrix = dist_df.pivot(index="query", columns="ref", values="dist")
dist_matrix = dist_matrix.reindex(index=acc_list, columns=acc_list)

# fill_diagonal requires a writable array  -  copy the underlying numpy array first
arr = dist_matrix.to_numpy().copy()
np.fill_diagonal(arr, 0.0)
dist_matrix = pd.DataFrame(arr, index=acc_list, columns=acc_list)

# Symmetrise (average both directions  -  should already be symmetric but enforce it)
dist_matrix = (dist_matrix + dist_matrix.T) / 2

print(f"Distance matrix shape: {dist_matrix.shape}")
print(f"Diagonal (should all be 0): min={np.diag(dist_matrix.values).min():.4f}")
print(f"Off-diagonal range: min={dist_matrix.values[dist_matrix.values > 0].min():.4f}, "
      f"max={dist_matrix.values.max():.4f}")
print(f"Median pairwise distance: {np.median(dist_matrix.values[np.triu_indices(len(acc_list), k=1)]):.4f}")

# Save compressed matrix for reuse
dist_matrix.to_parquet(MASH_DIR / "distance_matrix.parquet")
print("Saved: data/interim/mash/distance_matrix.parquet")

Loading distance file...


  Rows loaded: 770,884


Distance matrix shape: (878, 878)
Diagonal (should all be 0): min=0.0000
Off-diagonal range: min=0.0000, max=1.0000
Median pairwise distance: 1.0000
Saved: data/interim/mash/distance_matrix.parquet


## Section 6  -  Distance distribution: what do the numbers mean?

Before clustering, look at the distribution of pairwise distances.  
This tells you where natural breaks occur  -  clones, lineages, species  -  and guides
the dendrogram cut threshold.

In [6]:
upper_tri = dist_matrix.values[np.triu_indices(len(dist_matrix), k=1)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full distribution
axes[0].hist(upper_tri, bins=100, color="#4878d0", edgecolor="none", alpha=0.85)
axes[0].set_xlabel("Mash distance", fontsize=10)
axes[0].set_ylabel("Pair count", fontsize=10)
axes[0].set_title("All 384,753 pairwise distances", fontsize=11)
axes[0].set_yscale("log")
for x, lbl in [(0.01, "within-lineage"), (0.05, "within-species"), (0.15, "cross-species")]:
    axes[0].axvline(x, color="red", linewidth=0.8, linestyle="--")
    axes[0].text(x + 0.002, axes[0].get_ylim()[1] * 0.3, lbl,
                 fontsize=7, color="red", rotation=90, va="top")

# Zoomed: close pairs (< 0.15)  -  where clustering decisions matter
close = upper_tri[upper_tri < 0.15]
axes[1].hist(close, bins=100, color="#ee854a", edgecolor="none", alpha=0.85)
axes[1].set_xlabel("Mash distance (< 0.15)", fontsize=10)
axes[1].set_ylabel("Pair count", fontsize=10)
axes[1].set_title(f"Close pairs only (n={len(close):,})", fontsize=11)
axes[1].set_yscale("log")
for x in [0.005, 0.01, 0.02, 0.05]:
    axes[1].axvline(x, color="red", linewidth=0.8, linestyle="--")
    axes[1].text(x + 0.001, axes[1].get_ylim()[1] * 0.3, str(x),
                 fontsize=7, color="red", rotation=90, va="top")

plt.suptitle("Mash pairwise distance distribution  -  878 ESKAPE genomes", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_mash_distance_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/figures/phylo/01_mash_distance_distribution.png")

# Summary statistics at key thresholds
print("\nFraction of pairs at each distance threshold:")
for thr in [0.005, 0.01, 0.02, 0.05, 0.10, 0.15]:
    frac = (upper_tri <= thr).mean()
    n    = (upper_tri <= thr).sum()
    print(f"  <= {thr:.3f}: {n:6,} pairs  ({100*frac:.2f}% of all pairs)")

Saved: results/figures/phylo/01_mash_distance_distribution.png

Fraction of pairs at each distance threshold:
  <= 0.005:  4,586 pairs  (1.19% of all pairs)
  <= 0.010: 20,518 pairs  (5.33% of all pairs)
  <= 0.020: 43,825 pairs  (11.38% of all pairs)
  <= 0.050: 54,974 pairs  (14.28% of all pairs)
  <= 0.100: 56,985 pairs  (14.80% of all pairs)
  <= 0.150: 64,204 pairs  (16.68% of all pairs)


## Section 7  -  Why global clustering fails here, and the within-species fix

Global clustering with a single distance threshold failed for this dataset for a
concrete reason: SA, PA, and KP are highly clonal at this scale  -  almost all
within-species Mash distances are below 0.02. Global clustering at t=0.02 merges
all 150 SA genomes into a single phylogroup, and similarly collapses PA into 2
groups and KP into 2 groups.

**The consequence for 5-fold GroupedStratifiedKFold:**
When SA has only 1 phylogroup, all 150 SA genomes are assigned to the same CV fold
as a test set. The model must classify SA in that fold without ever having trained
on a single SA genome  -  effectively leave-one-species-out CV. This is not the goal.
Phylogenetic grouping should prevent *within-species* clone contamination, not hold
out entire species.

**The fix  -  within-species clustering:**
Run hierarchical clustering separately per species, using the same 878×878 distance
matrix (just subsetting to within-species pairs). Each species gets its own set of
phylogroups, ensuring every species contributes to all 5 folds.

**Threshold choice  -  t=0.010 within-species:**
At t=0.010, every species produces ≥8 groups (SA=10, PA=8, KP=28, EF=17, AB=35, EC=95),
giving all species representation across folds. A lower threshold (t=0.005) creates
too many singleton groups in EC (113 groups from 146 genomes). A higher threshold
(t=0.015) collapses KP to 3 groups and PA to 3 groups  -  too few for stable 5-fold CV.
t=0.010 is the empirical optimum for this dataset.

In [7]:
WITHIN_SP_THRESHOLD = 0.010   # default for all species

# PA-specific override: PA clinical strains have unusually low within-species
# diversity (max pairwise Mash distance = 0.026, median = 0.010). At t=0.010,
# average linkage merges 104 independently-evolved PA isolates (many different STs)
# into a single phylogroup, producing a 104-genome group that would dominate one
# CV fold. The genuine near-clone pairs in PA are at distance <0.005, so t=0.005
# is the biologically appropriate threshold for PA.
SPECIES_THRESHOLDS = {
    "paeruginosa": 0.005,    # tightened: prevents mega-group (original decision)
    "saureus":     0.005,    # tightened 2026-06-05: 67 STs in dominant group; concordance 97.7% at t=0.005
    "efaecium":    0.007,    # tightened 2026-06-05: 44 STs in dominant group; t=0.005 failed concordance (52.5%); t=0.007 is minimum passing threshold
}

dm = pd.read_parquet(MASH_DIR / "distance_matrix.parquet")

species_list = sorted(fm["species"].unique())
all_phylo = {}

print("Within-species hierarchical clustering")
print(f"Default threshold: t={WITHIN_SP_THRESHOLD}  |  PA override: t={SPECIES_THRESHOLDS['paeruginosa']}")
print(f"\n{'Species':<15}  {'t':>6}  {'N':>5}  {'Groups':>7}  {'Singletons':>11}  {'MaxSize':>8}")
print("-" * 62)

for sp in species_list:
    t        = SPECIES_THRESHOLDS.get(sp, WITHIN_SP_THRESHOLD)
    sp_accs  = fm[fm["species"] == sp].index.tolist()
    sub_dm   = dm.loc[sp_accs, sp_accs]
    cond     = squareform(sub_dm.values, checks=False)
    Z_sp     = linkage(cond, method="average")
    labels   = fcluster(Z_sp, t=t, criterion="distance")

    n_groups    = len(set(labels))
    group_sizes = pd.Series(labels).value_counts()
    n_singletons = (group_sizes == 1).sum()
    max_size     = group_sizes.max()

    sp_abbrev = sp[:2].upper()
    size_rank = group_sizes.rank(ascending=False, method="first").astype(int)
    for acc, lbl in zip(sp_accs, labels):
        all_phylo[acc] = f"{sp_abbrev}_PG_{size_rank[lbl]:03d}"

    print(f"{sp:<15}  {t:>6.3f}  {len(sp_accs):>5}  {n_groups:>7}  {n_singletons:>11}  {max_size:>8}")

phylo_series = pd.Series(all_phylo, name="phylogroup")
print(f"\nTotal phylogroups (before singleton merge): {phylo_series.nunique()}")

Within-species hierarchical clustering
Default threshold: t=0.01  |  PA override: t=0.005

Species               t      N   Groups   Singletons   MaxSize
--------------------------------------------------------------
abaumannii        0.010    150       35           22        76
ecloaceae         0.010    146       95           73         9
efaecium          0.010    150       17           10       114
kpneumoniae       0.010    132       28           10        28
paeruginosa       0.005    150       85           59        10
saureus           0.010    150       10            1       104

Total phylogroups (before singleton merge): 270


## Section 8  -  MLST concordance check

Within-species clustering should produce groups where same-ST genomes land in the
same phylogroup. We verify this before accepting the grouping for CV use.

**Expected result at t=0.010:**  
Species with dominant clones (SA, PA) should show high concordance  -  same ST means
near-identical genomes, and near-identical means distance < 0.010.  
EC may have lower concordance because the complex comprises 6 genomospecies with
different MLST schemes, and some within-ST distances may exceed 0.010.

In [8]:
check_df = phylo_series.to_frame().join(fm[["species", "sequence_type"]])

concordant_total = 0
discordant_total = 0
sts_total = 0

print(f"{'Species':<15}  {'STs≥2':>6}  {'Concordant':>11}  {'%':>6}")
print("-" * 45)

for sp in species_list:
    sp_check = check_df[check_df["species"] == sp]
    st_groups = sp_check[sp_check["sequence_type"].notna()].groupby("sequence_type")
    conc = disc = n_sts = 0
    for st, grp in st_groups:
        if len(grp) < 2:
            continue
        n_sts += 1
        if grp["phylogroup"].nunique() == 1:
            conc += 1
        else:
            disc += 1
    pct = 100 * conc / n_sts if n_sts > 0 else 0.0
    print(f"{sp:<15}  {n_sts:>6}  {conc:>11}  {pct:>5.1f}%")
    concordant_total += conc
    discordant_total += disc
    sts_total += n_sts

pct_overall = 100 * concordant_total / sts_total if sts_total > 0 else 0
print(f"\n{'OVERALL':<15}  {sts_total:>6}  {concordant_total:>11}  {pct_overall:>5.1f}%")

if pct_overall >= 90:
    print("\nPASS: >=90% MLST concordance.")
else:
    print(f"\nWARNING: {pct_overall:.1f}% concordance < 90% target.")
    print("Discordant STs = same-ST genomes in different phylogroups (threshold may be too fine).")
    print("This is acceptable if discordant STs have genuine within-ST diversity > 0.010.")

CHOSEN_THRESHOLD = WITHIN_SP_THRESHOLD
pct_concordant = pct_overall

Species           STs≥2   Concordant       %
---------------------------------------------
abaumannii           10           10  100.0%
ecloaceae            19           19  100.0%
efaecium             19           18   94.7%
kpneumoniae          20           20  100.0%
paeruginosa          20           20  100.0%
saureus              21           21  100.0%

OVERALL             109          108   99.1%

PASS: >=90% MLST concordance.


## Section 10  -  Visualisation: phylogroup size distribution and species composition

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: phylogroup size histogram
sizes = phylo_series.value_counts().values
axes[0].hist(sizes, bins=40, color="#4878d0", edgecolor="white", linewidth=0.4)
axes[0].axvline(1, color="red", linewidth=1.2, linestyle="--", label="Singleton")
axes[0].set_xlabel("Phylogroup size (genomes)", fontsize=10)
axes[0].set_ylabel("Number of phylogroups", fontsize=10)
axes[0].set_title(f"Phylogroup size distribution\n"
                   f"(threshold={CHOSEN_THRESHOLD}, n={phylo_series.nunique()} groups)", fontsize=11)
axes[0].legend(fontsize=9)

# Right: species composition of top-20 largest phylogroups
sp_colors = {
    "abaumannii": "#4878d0", "efaecium": "#ee854a",
    "kpneumoniae": "#6acc65", "paeruginosa": "#d65f5f",
    "saureus": "#956cb4", "ecloaceae": "#8c613c",
}
top20 = phylo_series.value_counts().head(20).index
comp = check_df[check_df["phylogroup"].isin(top20)].groupby(
    ["phylogroup", "species"]).size().unstack(fill_value=0)
comp = comp.reindex(top20)
comp.plot(kind="bar", stacked=True, color=[sp_colors.get(c, "grey") for c in comp.columns],
          ax=axes[1], width=0.8, edgecolor="none")
axes[1].set_xlabel("Phylogroup (top 20 by size)", fontsize=10)
axes[1].set_ylabel("Genome count", fontsize=10)
axes[1].set_title("Species composition of 20 largest phylogroups", fontsize=11)
axes[1].legend(title="Species", fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
axes[1].tick_params(axis="x", rotation=45, labelsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "02_phylogroup_composition.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/figures/phylo/02_phylogroup_composition.png")

Saved: results/figures/phylo/02_phylogroup_composition.png


## Section 11  -  Handle singletons

Singletons (phylogroups of size 1) cannot be stratified across folds properly.
Strategy: merge each singleton with its nearest non-singleton phylogroup
(by minimum Mash distance to any member of that group).

This is conservative  -  it assigns the singleton to its closest existing group
rather than creating an artificial cluster.

In [10]:
pg_counts = phylo_series.value_counts()
singleton_labels = pg_counts[pg_counts == 1].index.tolist()
singleton_accs   = [acc for acc in phylo_series.index
                    if phylo_series[acc] in singleton_labels]

print(f"Singletons before merging: {len(singleton_accs)}")

phylo_series_final = phylo_series.copy()

if singleton_accs:
    for s_acc in singleton_accs:
        sp = fm.loc[s_acc, "species"]
        # Candidate targets: non-singleton genomes of the same species
        sp_accs   = fm[fm["species"] == sp].index.tolist()
        non_sing  = [a for a in sp_accs
                     if a != s_acc and phylo_series_final[a] not in singleton_labels]
        if not non_sing:
            # All same-species genomes are singletons  -  keep as-is
            continue
        # Merge into nearest non-singleton same-species group
        dists      = dm.loc[s_acc, non_sing]
        nearest    = dists.idxmin()
        target_pg  = phylo_series_final[nearest]
        phylo_series_final[s_acc] = target_pg

    n_singletons_after = (phylo_series_final.value_counts() == 1).sum()
    print(f"Singletons after merging:  {n_singletons_after}")

phylo_series_final.name = "phylogroup"
print(f"\nFinal phylogroup count:  {phylo_series_final.nunique()}")
print(f"Smallest group size:     {phylo_series_final.value_counts().min()}")
print(f"Largest group size:      {phylo_series_final.value_counts().max()}")
print(f"\nGroups per species:")
for sp in species_list:
    sp_accs = fm[fm["species"]==sp].index
    n_pg = phylo_series_final[sp_accs].nunique()
    print(f"  {sp:<15}: {n_pg} phylogroups")

Singletons before merging: 175

Singletons after merging:  0

Final phylogroup count:  95
Smallest group size:     2
Largest group size:      119

Groups per species:
  abaumannii     : 13 phylogroups
  ecloaceae      : 22 phylogroups
  efaecium       : 7 phylogroups
  kpneumoniae    : 18 phylogroups
  paeruginosa    : 26 phylogroups
  saureus        : 9 phylogroups


## Section 12  -  Save outputs

Two outputs:
1. **`cv_groups.parquet`**  -  the phylogroup series indexed by accession. This is
   imported directly by Phase 7 and all subsequent notebooks as the CV grouping variable.
2. **`feature_matrix.parquet`**  -  updated with the `phylogroup` column added.

In [11]:
# 1. Save cv_groups
cv_groups_path = PROC / "cv_groups_3460.parquet"
phylo_series_final.to_frame().to_parquet(cv_groups_path)
print(f"Saved: {cv_groups_path}")

# 2. Update feature matrix
fm_updated = fm.copy()
fm_updated["phylogroup"] = phylo_series_final
assert fm_updated["phylogroup"].isna().sum() == 0, "Some genomes have no phylogroup assigned!"
fm_updated.to_parquet(PROC / "feature_matrix_3460.parquet")
print(f"Saved: feature_matrix_3460.parquet (with phylogroup column)")
print(f"  New column 'phylogroup' added. Shape: {fm_updated.shape}")

# Quick final summary
print("\n=== Phase 6 complete ===")
print(f"Distance threshold used:     {CHOSEN_THRESHOLD}")
print(f"Phylogroups assigned:         {phylo_series_final.nunique()}")
print(f"Singletons (before merging):  {len(singleton_accs)}")
print(f"MLST concordance:             {pct_concordant:.1f}%")
print(f"Cross-species contamination:  0%")
print()
print("Next: Phase 7  -  Baseline classifiers with GroupedStratifiedKFold")
print("  Import: from data/processed/cv_groups.parquet")

Saved: ../data/processed/cv_groups.parquet
Saved: ../data/processed/feature_matrix.parquet
  New column 'phylogroup' added. Shape: (878, 632)

=== Phase 6 complete ===
Distance threshold used:     0.01
Phylogroups assigned:         95
Singletons (before merging):  175
MLST concordance:             99.1%
Cross-species contamination:  0%

Next: Phase 7  -  Baseline classifiers with GroupedStratifiedKFold
  Import: from data/processed/cv_groups.parquet
